# Custom quadruped — Colab training

Runtime → Change runtime type → **T4 GPU** (or any GPU).

Opening this notebook from GitHub does **not** copy the rest of the repo. The next cell clones [jakhon37/quad-loco](https://github.com/jakhon37/quad-loco) into `/content/quad-loco`.

Physics runs on CPU (8 processes). The GPU runs PPO.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = 'https://github.com/jakhon37/quad-loco.git'
candidates = [
    Path('/content/quad-loco'),
    Path.cwd(),
    Path.cwd() / 'quad-loco',
    Path.cwd().parent,
]
ROOT = next((p for p in candidates if (p / 'src' / 'quad_loco').is_dir()), None)
if ROOT is None:
    ROOT = Path('/content/quad-loco') if Path('/content').exists() else Path.cwd() / 'quad-loco'
    print(f'Cloning {REPO} -> {ROOT}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, str(ROOT)])
elif (ROOT / '.git').is_dir():
    print(f'Updating {ROOT}')
    subprocess.check_call(['git', '-C', str(ROOT), 'fetch', 'origin', 'main'])
    subprocess.check_call(['git', '-C', str(ROOT), 'reset', '--hard', 'origin/main'])
print('ROOT', ROOT)
assert (ROOT / 'src' / 'quad_loco').is_dir(), f'repo missing at {ROOT}'
os.chdir(ROOT)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# Colab ships the unmaintained `gym` package; we use Gymnasium.
!pip -q uninstall -y gym
!pip -q install -r requirements.txt
!apt-get -qq update && apt-get -qq install -y libosmesa6 libosmesa6-dev libegl1 >/dev/null
import torch, mujoco
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('mujoco', mujoco.__version__, 'MUJOCO_GL', os.environ.get('MUJOCO_GL'))

In [ ]:
from quad_loco.convert_urdf import convert
from quad_loco.paths import scene_xml
xml = convert()
print('scene', xml)
import mujoco
m = mujoco.MjModel.from_xml_path(str(xml))
print(f'nq={m.nq} nu={m.nu}')

In [ ]:
from quad_loco.env import QuadrupedVelocityEnv
env = QuadrupedVelocityEnv(easy=True, command=(0.4, 0.0, 0.0))
obs, info = env.reset(seed=0)
print('obs', obs.shape, 'cmd', info['command'])
for _ in range(20):
    obs, r, term, trunc, info = env.step(env.action_space.sample())
print('height', info['base_height'], 'reward', r)
env.close()

In [ ]:
# SMOKE=True: 8192 steps (~2 PPO updates, a few minutes) to prove the pipeline.
# SMOKE=False: 1.5 million steps, 2-4 hours. That is the run that can learn a walk.
SMOKE = True
extra = ' --timesteps 8192' if SMOKE else ''
cmd = 'python scripts/train.py --config configs/ppo_colab.yaml --run-name colab_easy' + extra
print(cmd)
!{cmd}

In [ ]:
from pathlib import Path
from quad_loco.checkpoints import resolve_sb3_zip

try:
    model = resolve_sb3_zip(Path('logs/colab_easy/final_model.zip'))
except FileNotFoundError as exc:
    print(exc)
    print('Train the previous cell first. For a full walk, re-run train without --timesteps.')
else:
    print('using', model)
    !python scripts/eval.py --model logs/colab_easy/final_model --easy --command 0.5 0 0 --video videos/walk.mp4 --episodes 1 --max-seconds 8
    !python scripts/export_onnx.py --model logs/colab_easy/final_model --out logs/colab_easy/policy.onnx
    from IPython.display import Image, display
    from google.colab import files
    for p in ['videos/walk_preview.png', 'videos/walk.gif']:
        if Path(p).exists():
            display(Image(p))
    for p in ['logs/colab_easy/final_model.zip', 'logs/colab_easy/vecnormalize.pkl', 'logs/colab_easy/policy.onnx', 'logs/colab_easy/progress.csv', 'videos/walk.mp4', 'videos/walk_preview.png']:
        if Path(p).exists():
            files.download(p)